# AN8 Part 3 — Semantic Analysis and Controlled Transformations

**Status:** Prepared notebook; not executed.  
**Purpose:** Review retained P3 SQL result exports, reconcile evidence, document semantic decisions, and implement only justified, reversible Python transformations after the diagnostic gates pass.

## Operating boundaries

- Notebook 10 remains the AN8 Part 2 execution and review record.
- Canonical SQL remains under sql/05_part3; this notebook does not duplicate or execute SQL.
- P3 exports are read from outputs/an8/part3 and verified against their execution manifests.
- The full student_vle source is never loaded into one Pandas DataFrame. Any later row-level processing must use chunked reads or database-side aggregation.
- Original source files and PostgreSQL RAW, QUALITY, and STAGING are immutable.
- No transformation is approved merely because code can perform it. Preserve raw values and document the evidence, affected keys, row counts, alternatives, and metric impact first.
- No prediction, ML training, or Power BI work is performed in this notebook.

## 00 — Environment and project paths

The first code cell locates the repository root and imports only the libraries needed to inspect exported result sets. It does not connect to PostgreSQL.

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "sql" / "05_part3").is_dir() and (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the OULAD project root from the notebook directory.")

PROJECT_ROOT = find_project_root()
PART3_OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "an8" / "part3"

print(f"Project root: {PROJECT_ROOT}")
print(f"Part 3 output root: {PART3_OUTPUT_ROOT}")
print(f"Part 3 output directory exists: {PART3_OUTPUT_ROOT.is_dir()}")

## 01 — Load and verify one query execution

Each invocation of scripts/run_part3.py runs exactly one query and creates a timestamped folder containing result_set_XX.csv files and execution_manifest.json. The helper below verifies the exported file hashes before loading them. Use it only for result sets small enough to fit in memory; P3.04 and other larger outputs may require chunked CSV reading.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


def find_query_manifests(query_id: str) -> list[Path]:
    return sorted(
        PART3_OUTPUT_ROOT.glob(f"*/{query_id}/execution_manifest.json"),
        key=lambda path: path.parent.parent.name,
    )


def load_query_results(query_id: str, run_directory: Path | None = None):
    if run_directory is None:
        manifests = find_query_manifests(query_id)
        if not manifests:
            raise FileNotFoundError(f"No retained execution manifest found for {query_id}.")
        manifest_path = manifests[-1]
    else:
        manifest_path = Path(run_directory) / query_id / "execution_manifest.json"

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("query_id") != query_id or manifest.get("read_only") is not True:
        raise ValueError(f"Unexpected query identity or read-only status in {manifest_path}")

    frames = {}
    for item in manifest["result_sets"]:
        csv_path = manifest_path.parent / item["file"]
        if sha256_file(csv_path) != item["sha256"]:
            raise ValueError(f"Export hash mismatch: {csv_path}")
        frame = pd.read_csv(csv_path, low_memory=False)
        if len(frame) != item["rows"]:
            raise ValueError(f"Export row-count mismatch: {csv_path}")
        frames[item["result_set"]] = frame

    return manifest, frames


print("Available P3.00 manifests:", len(find_query_manifests("P3.00")))
print("Available P3.07 manifests:", len(find_query_manifests("P3.07")))

## 02 — P3.00 source-import reconciliation

Load P3.00 only after its local execution has completed. Compare the exact imported row counts with the approved source inventory: courses 22, assessments 206, student_info 32,593, student_registration 32,593, student_assessment 173,912, vle 6,364, and student_vle 10,655,280. Any mismatch blocks further analysis.

In [ ]:
EXPECTED_SOURCE_ROWS = {
    "courses": 22,
    "assessments": 206,
    "student_info": 32593,
    "student_registration": 32593,
    "student_assessment": 173912,
    "vle": 6364,
    "student_vle": 10655280,
}

p300_manifest = None
p300_results = None

if find_query_manifests("P3.00"):
    p300_manifest, p300_results = load_query_results("P3.00")
    print("Execution metadata:")
    display(pd.Series({
        "executed_at_utc": p300_manifest["executed_at_utc"],
        "elapsed_seconds": p300_manifest["elapsed_seconds"],
        "database": p300_manifest["database"]["database"],
        "read_only": p300_manifest["read_only"],
    }).to_frame("value"))

    for result_number, frame in p300_results.items():
        print(f"P3.00 result set {result_number}")
        display(frame)
else:
    print("P3.00 NOT AVAILABLE — no SQL execution is inferred.")

## 03 — P3.07 grain and join gate

P3.07 must be reviewed before interpreting any cross-domain join. Inspect duplicate-key counts, registration/profile key-set differences, join row counts, and the activity reference-match matrix. A duplicate count is evidence of a key issue, not permission to delete records.

In [ ]:
p307_manifest = None
p307_results = None

if find_query_manifests("P3.07"):
    p307_manifest, p307_results = load_query_results("P3.07")
    print("Execution metadata:")
    display(pd.Series({
        "executed_at_utc": p307_manifest["executed_at_utc"],
        "elapsed_seconds": p307_manifest["elapsed_seconds"],
        "database": p307_manifest["database"]["database"],
        "read_only": p307_manifest["read_only"],
    }).to_frame("value"))

    for result_number, frame in p307_results.items():
        print(f"P3.07 result set {result_number}")
        display(frame)
else:
    print("P3.07 NOT AVAILABLE — the cross-domain gate remains unreviewed.")

## 04 — P3.04 zero-activity cohort reconciliation

P3.04 returns one row per registered student-presentation after a single VLE aggregation. The Python summaries below classify the complete cohort and test the historically observed 3,365 no-recorded-row and 4,093 no-in-course-activity counts. The 728 difference is treated as a reconciliation target, not assumed to be pre-course-only until the row-level states demonstrate it.

In [ ]:
p304_manifest = None
p304_results = None
p304_cohort = None

if find_query_manifests("P3.04"):
    p304_manifest, p304_results = load_query_results("P3.04")
    if len(p304_results) != 1:
        raise ValueError("P3.04 must return exactly one complete cohort result set.")
    p304_cohort = p304_results[1]

    required_columns = {
        "code_module", "code_presentation", "id_student",
        "all_activity_rows", "pre_rows", "in_rows", "post_rows",
        "unclassified_rows", "activity_state",
    }
    missing = required_columns - set(p304_cohort.columns)
    if missing:
        raise ValueError(f"P3.04 missing required columns: {sorted(missing)}")

    cohort_key = ["code_module", "code_presentation", "id_student"]
    print("P3.04 row count:", len(p304_cohort))
    print("Distinct registered keys:", p304_cohort[cohort_key].drop_duplicates().shape[0])
    print("Duplicate output keys:", p304_cohort.duplicated(cohort_key).sum())

    state_summary = (
        p304_cohort.groupby("activity_state", dropna=False)
        .size().rename("student_presentations").reset_index()
        .sort_values("activity_state")
    )
    display(state_summary)

    historical_comparison = pd.DataFrame([
        {
            "measure": "No recorded VLE row",
            "historical_AN8": 3365,
            "P3.04_source_ref": int((p304_cohort["all_activity_rows"] == 0).sum()),
        },
        {
            "measure": "No in-course VLE row",
            "historical_AN8": 4093,
            "P3.04_source_ref": int((p304_cohort["in_rows"] == 0).sum()),
        },
        {
            "measure": "Difference",
            "historical_AN8": 728,
            "P3.04_source_ref": int((p304_cohort["in_rows"] == 0).sum())
                - int((p304_cohort["all_activity_rows"] == 0).sum()),
        },
    ])
    display(historical_comparison)

    print("No-in-course records by exact activity state:")
    display(
        p304_cohort.loc[p304_cohort["in_rows"] == 0]
        .groupby("activity_state", dropna=False)
        .size().rename("student_presentations").reset_index()
    )

    print("Presentation/outcome/state breakdown:")
    display(
        p304_cohort.groupby(
            ["code_module", "code_presentation", "final_result", "activity_state"],
            dropna=False,
        ).size().rename("student_presentations").reset_index()
    )
else:
    print("P3.04 NOT AVAILABLE — the 728-record difference remains unverified.")

## 05 — Evidence review and decision register

No transformation is implemented in this section. Record only decisions supported by reviewed P3 outputs and source documentation. Keep unresolved items explicitly open.

## 05 — Controlled Python processing design

The source CSVs remain immutable. Any later row-level processing of studentVle.csv must use pandas.read_csv(..., chunksize=...) or a similarly bounded streaming approach. The complete file must not be loaded into memory. This section remains a design gate until P3.01–P3.07 have been reviewed and the relevant semantic rules have been documented.

A future transformation must:
1. Preserve the original source columns and values.
2. Add separately named typed/semantic fields rather than overwriting source fields.
3. Write to a versioned output location with a manifest.
4. Reconcile source rows, output rows, key counts, missing tokens, and numeric totals.
5. Compare alternative scenarios and document downstream metric changes.
6. Remain reproducible and reversible.

## 06 — Reporting contract gate

No Power BI dataset is prepared until the reporting contract is approved. Every candidate indicator must define its business question, grain, population, numerator, denominator, time basis, NULL/zero handling, activity scenario, join path, lineage, validation, and limitations.

The final analytical report follows Power BI. Prediction and machine learning are deferred to a separate Data Science phase after Power BI and are outside AN8.